# ESN benchmark: feature sets and sequence lengths

Purpose:

1. Use the classical diagnostics to define cleaner ESN inputs.
2. Test whether sequence/reservoir dynamics improve over simple tabular baselines.
3. Compare compact, correlation-pruned, PCA, hand-selected, and expanded inputs.
4. Aggregate across random seeds instead of selecting a lucky seed.

This notebook is intended to be run cell by cell. Start small, inspect results, then expand the grid only if useful.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

print(Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.data.loaders import load_market_stress_data
from qpitome_qrc.baselines.esn import grid_configs
from qpitome_qrc.baselines.esn_benchmark import (
    default_esn_feature_sets,
    run_esn_benchmark_suite,
    aggregate_esn_seeds,
    save_esn_benchmark_outputs,
)

## 1. Load data

In [ ]:
df = load_market_stress_data()
print(df.shape)
df.head()

## 2. Define ESN feature sets

These are derived from the classical diagnostic report:

- `compact`: original compact 12 features
- `hand_selected`: low-redundancy domain features
- `compact_corr_pruned_0.95`: train-fitted correlation pruning
- `compact_pca_6`: train-fitted PCA compression
- `expanded`: compact + exploratory multi-scale features, if present

For QRC, `compact_pca_6` and `hand_selected` are especially relevant because they constrain input dimension.

In [ ]:
feature_sets = default_esn_feature_sets(df, pca_components=6, corr_threshold=0.95)

for fs in feature_sets:
    print(fs.name, len(fs.features), fs.features, "transformer=", fs.transformer_builder is not None)

## 3. Minimal first ESN benchmark run

This is intentionally small. It tests feature-set and sequence-length effects before doing broader hyperparameter tuning.

Current grid:

- feature sets: all default feature sets
- sequence lengths: 20, 40
- units: 300, 600
- seeds: 1, 2, 3
- fixed spectral radius/leak/connectivity/readout based on earlier best-ish region

Total runs should be manageable.

In [ ]:
configs = grid_configs(
    units=(300, 600),
    spectral_radius=(0.7,),
    leak_rate=(0.5,),
    reservoir_connectivity=(0.1,),
    readout_C=(0.1,),
    seeds=(1, 2, 3),
    washout=(0,),
    pooling=("final",),
    scale_states=(False,),
)

print("n_configs:", len(configs))
configs[:3]

In [ ]:
summary, results = run_esn_benchmark_suite(
    df=df,
    feature_sets=feature_sets,
    seq_lens=(20, 40),
    configs=configs,
)

summary.head(20)

## 4. Aggregate across seeds

Use this table for model selection, not individual seed rows.

In [ ]:
agg = aggregate_esn_seeds(summary)
agg.head(20)

In [ ]:
cols = [
    "feature_set",
    "seq_len",
    "n_input_features",
    "units",
    "mean_val_pr_auc",
    "std_val_pr_auc",
    "mean_val_f1",
    "mean_test_pr_auc",
    "n_seeds",
]
agg[cols].head(20)

## 5. Visual comparison: feature sets and window lengths

In [ ]:
plot_df = agg.sort_values("mean_val_pr_auc", ascending=True).tail(20).copy()
labels = (
    plot_df["feature_set"]
    + " | T=" + plot_df["seq_len"].astype(str)
    + " | N=" + plot_df["units"].astype(str)
)

plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
plt.barh(labels, plot_df["mean_val_pr_auc"], xerr=plot_df["std_val_pr_auc"].fillna(0))
plt.xlabel("Mean validation PR-AUC across seeds")
plt.title("ESN benchmark: validation PR-AUC by feature set/window")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
pivot = agg.pivot_table(
    index="feature_set",
    columns="seq_len",
    values="mean_val_pr_auc",
    aggfunc="max",
)
pivot

In [ ]:
plt.figure(figsize=(7, 4))
for fs in pivot.index:
    plt.plot(pivot.columns, pivot.loc[fs], marker="o", label=fs)
plt.xlabel("Sequence length")
plt.ylabel("Best mean validation PR-AUC")
plt.title("Sequence length effect by feature set")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Compare ESN to toy tabular benchmark target

From the classical report, the best toy tabular validation PR-AUC was about `0.578`.

For ESN to be useful as a serious classical temporal benchmark, we want it to clearly exceed that or at least reach roughly `>0.60` validation PR-AUC with stable seed behavior.

In [ ]:
TABULAR_REFERENCE_VAL_PR_AUC = 0.578
TARGET_ESN_VAL_PR_AUC = 0.60

agg.assign(
    beats_tabular_reference=agg["mean_val_pr_auc"] > TABULAR_REFERENCE_VAL_PR_AUC,
    reaches_esn_target=agg["mean_val_pr_auc"] > TARGET_ESN_VAL_PR_AUC,
)[cols + ["beats_tabular_reference", "reaches_esn_target"]].head(30)

## 7. Save benchmark outputs

This saves CSV tables for later report generation.

In [ ]:
out_dir = save_esn_benchmark_outputs(
    summary=summary,
    aggregate=agg,
    output_dir="reports/esn_benchmark/tables",
)
print(out_dir)

## 8. Interpretation scratchpad

Questions to answer after the first run:

1. Does any feature set exceed the tabular reference?
2. Does PCA-6 help or hurt ESN?
3. Does hand-selected low-redundancy input help?
4. Does sequence length 40 beat 20?
5. Is seed variance small enough to trust the result?
6. If ESN still fails, do we move to LSTM rather than keep tuning?

Notes:

- A stronger readout can be tested later, but if reservoir features are not separable, the readout mostly moves the precision/recall tradeoff.
- QRC pitch remains plausible: a quantum reservoir may create a richer feature map than the ESN, but only if we control the classical benchmark honestly.
- If PCA-6 is competitive, it is a good QRC input candidate because it gives compact, orthogonal, low-redundancy inputs.